In [2]:
import pandas as pd
import logging
import sys
from pathlib import Path
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[ 
        logging.FileHandler("pipeline.log", encoding="utf-8"), # create file
        logging.StreamHandler(sys.stdout),   # write terminal
    ],
)
logger = logging.getLogger(__name__)
logger.info("Pipeline khởi động.")

2026-04-28 16:50:27,418 | INFO | Pipeline khởi động.


In [3]:
class OrderSchema(BaseModel):
    order_id:                 str
    customer_id:              str
    order_status:             str
    order_purchase_timestamp: str
    customer_state:           str
    price:                    float
    freight_value:            float
    payment_value:            Optional[float] = None  # don't pay
    product_id:               Optional[str]   = None  # don't purchase

    @field_validator("price", "freight_value") # @field_validator registers a function to validate specific fields; 
    @classmethod                                
    def non_negative(cls, v):                  # only decorated functions run, and multiple validators run in order.
        if v < 0:
            raise ValueError(f"Không được âm: {v}") 
        return v

    @field_validator("order_status")
    @classmethod
    def valid_status(cls, v):
        allowed = {
            "delivered","shipped","canceled",
            "unavailable","processing",
            "invoiced","approved","created",
        }
        if v not in allowed:
            raise ValueError(f"Status lạ: {v}")
        return v

print("✅ Schema OK")

✅ Schema OK


In [4]:
def extract(path: str) -> pd.DataFrame:
    logger.info(f"[EXTRACT] Đọc file: {path}")
    try:
        df = pd.read_csv(path, low_memory=False)           # read whole instead of each chunk
        logger.info(f"[EXTRACT] Xong — shape: {df.shape}")
        return df
    except FileNotFoundError:
        logger.error(f"[EXTRACT] Không tìm thấy: {path}")
        raise

In [ ]:
REQUIRED_COLS = [
    "order_id","customer_id","order_status",
    "order_purchase_timestamp","customer_state",
    "price","freight_value",
]

def validate(df: pd.DataFrame, error_path: str = "errors.csv") -> pd.DataFrame:
    logger.info("[VALIDATE] Bắt đầu...")

    missing = [c for c in REQUIRED_COLS if c not in df.columns]     # [] : false, ["customer_id","order_status"] : true
    if missing:
        raise ValueError(f"Thiếu columns: {missing}")

    validate_cols = REQUIRED_COLS + ["payment_value","product_id"]
    validate_cols = [c for c in validate_cols if c in df.columns]

    valid_idx, errors = [], []

    # for idx, row in df[validate_cols].iterrows():              iterrows : read a line
    #     row_dict = {k: (None if pd.isna(v) else v) for k, v in row.items()}
    for record in df[validate_cols].to_dict('records'):
        row_dict = {k: (None if pd.isna(v) else v) for k, v in record.items()}    
        try:
            OrderSchema(**row_dict)
            valid_idx.append(idx)
        except ValidationError as e:
            row_dict["_error"] = str(e.errors()[0]["msg"])
            errors.append(row_dict)

    if errors:
        pd.DataFrame(errors).to_csv(error_path, index=False)
        logger.warning(f"[VALIDATE] {len(errors)} lỗi → {error_path}")

    df_valid = df.loc[valid_idx].reset_index(drop=True)
    logger.info(
        f"[VALIDATE] Hợp lệ: {len(df_valid):,}/{len(df):,} "
        f"({len(df_valid)/len(df)*100:.1f}%)"
    )
    return df_valid

In [6]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    logger.info(f"[TRANSFORM] Input: {df.shape}")
    try:
        before = len(df)
        df = df.drop_duplicates(subset=["order_id","product_id"]).copy()
        logger.info(f"[TRANSFORM] Dedup xóa: {before - len(df):,} dòng")

        df["price"]         = df["price"].fillna(0.0)
        df["freight_value"] = df["freight_value"].fillna(0.0)

        df["order_purchase_timestamp"] = pd.to_datetime(
            df["order_purchase_timestamp"], errors="coerce"
        )
        df["year_month"] = df["order_purchase_timestamp"].dt.to_period("M")

        logger.info(f"[TRANSFORM] Output: {df.shape}")
        return df
    except Exception as e:
        logger.error(f"[TRANSFORM] Lỗi: {e}")
        raise


def flag_outliers(df: pd.DataFrame, col: str, n_std: float = 3.0) -> pd.DataFrame:
    mean = df[col].mean()
    std  = df[col].std()
    df   = df.copy()
    df[f"{col}_is_outlier"] = (df[col] - mean).abs() > n_std * std
    count = df[f"{col}_is_outlier"].sum()
    logger.info(f"[OUTLIER] '{col}': {count:,} outliers")
    return df


def calc_top_states(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    # nunique(order_id) vì dataset là item-level
    # count() sẽ đếm items, không phải orders
    df_states = (
        df.groupby("customer_state")
        .agg(
            total_orders  = ("order_id", "nunique"),
            total_revenue = ("price",    "sum"),      # price không phải payment_value
        )
        .reset_index()
        .sort_values("total_revenue", ascending=False)
        .head(top_n)
    )
    logger.info(f"[TOP_STATES] shape: {df_states.shape}")
    return df_states


def calc_revenue_growth(df: pd.DataFrame) -> pd.DataFrame:
    df_monthly = (
        df.groupby("year_month")
        .agg(
            total_orders  = ("order_id", "nunique"),
            total_revenue = ("price",    "sum"),
        )
        .reset_index()
        .sort_values("year_month")
    )
    df_monthly["revenue_growth_pct"] = (
        df_monthly["total_revenue"]
        .pct_change().mul(100).round(2).fillna(0)   # pct_change() : (current - previous) / previous
    )
    df_monthly["year_month"] = df_monthly["year_month"].astype(str)
    logger.info(f"[GROWTH] {len(df_monthly)} tháng")
    return df_monthly

In [7]:
def load(df: pd.DataFrame, out_path: str) -> None:
    logger.info(f"[LOAD] Ghi: {out_path}")
    try:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(out_path, index=False, engine="pyarrow")  # pandas create index (no business)
        kb = Path(out_path).stat().st_size / 1024               # byte -> KB
        logger.info(f"[LOAD] ✅ {out_path} ({len(df):,} rows, {kb:.1f} KB)")
    except Exception as e:
        logger.error(f"[LOAD] Lỗi: {e}")
        raise

In [8]:
INPUT_PATH         = "olist_sales.csv"
OUT_TOP_STATES     = "top_states_revenue.parquet"
OUT_MONTHLY_GROWTH = "monthly_growth.parquet"

logger.info("=" * 50)
logger.info("PIPELINE BẮT ĐẦU")
logger.info("=" * 50)

# Extract
df_raw = extract(INPUT_PATH)

# Validate — cell này chạy ~2-3 phút với 56MB
df_valid = validate(df_raw)

# Transform
df_clean = transform(df_valid)
df_clean = flag_outliers(df_clean, col="price")

# Aggregate
df_states  = calc_top_states(df_clean)
df_monthly = calc_revenue_growth(df_clean)

print(df_states)
print(df_monthly.head())

# Load
load(df_states,  OUT_TOP_STATES)
load(df_monthly, OUT_MONTHLY_GROWTH)

logger.info("PIPELINE HOÀN THÀNH ✅")

2026-04-28 16:51:40,874 | INFO | ==================================================
2026-04-28 16:51:40,875 | INFO | PIPELINE BẮT ĐẦU
2026-04-28 16:51:40,875 | INFO | ==================================================
2026-04-28 16:51:40,876 | INFO | [EXTRACT] Đọc file: olist_sales.csv
2026-04-28 16:51:42,183 | INFO | [EXTRACT] Xong — shape: (113390, 39)
2026-04-28 16:51:42,184 | INFO | [VALIDATE] Bắt đầu...
2026-04-28 16:51:48,401 | INFO | [VALIDATE] Hợp lệ: 113,390/113,390 (100.0%)
2026-04-28 16:51:48,403 | INFO | [TRANSFORM] Input: (113390, 39)
2026-04-28 16:51:48,505 | INFO | [TRANSFORM] Dedup xóa: 14,612 dòng
2026-04-28 16:51:48,545 | INFO | [TRANSFORM] Output: (98778, 40)
2026-04-28 16:51:48,557 | INFO | [OUTLIER] 'price': 1,775 outliers
2026-04-28 16:51:48,602 | INFO | [TOP_STATES] shape: (10, 3)
2026-04-28 16:51:48,629 | INFO | [GROWTH] 22 tháng
   customer_state  total_orders  total_revenue
25             SP         39951     4693225.19
18             RJ         12169     1621

In [9]:
# Đọc lại để CONFIRM không bị corrupt
df_check = pd.read_parquet("top_states_revenue.parquet")
print(df_check)
print(df_check.dtypes)

df_check2 = pd.read_parquet("monthly_growth.parquet")
print(df_check2)

  customer_state  total_orders  total_revenue
0             SP         39951     4693225.19
1             RJ         12169     1621426.02
2             MG         11188     1442787.93
3             RS          5266      672428.46
4             PR          4855      611361.92
5             SC          3492      465320.53
6             BA          3215      452826.75
7             DF          2050      280051.73
8             GO          1911      253378.69
9             ES          1978      250339.65
customer_state        str
total_orders        int64
total_revenue     float64
dtype: object
   year_month  total_orders  total_revenue  revenue_growth_pct
0     2016-10           268       38017.67                0.00
1     2016-12             1          10.90              -99.97
2     2017-01           737       99149.16           909525.32
3     2017-02          1592      212400.16              114.22
4     2017-03          2496      333704.94               57.11
5     2017-04          2